In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\SM24519\OneDrive - Cardiff Metropolitan University\Desktop\Data\Papers\Presentations and directions\Reviews\Year 3\final_phase_documents\tis_replica_v3\output")

In [ ]:
# individual systems (each source working independently) vs the TIS bundle (accumulated view)
individual = {}
tis_bundle = {}

for f in DATA_DIR.glob("*.csv"):
    df = pd.read_csv(f)
    if f.stem.startswith("patient_history"):
        tis_bundle[f.stem] = df
    else:
        individual[f.stem] = df

for name, df in {**individual, **tis_bundle}.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols")

## Step 2: unit normalization + time alignment

Individual systems report the same concept under different names/units (India "Total Cholesterol" mg/dl vs UK "Cholesterol" mmol/L — these even carry different LOINC codes, since LOINC is unit-specific). This maps everything to one canonical name/unit and one parsed timestamp so sources can be compared directly.

In [ ]:
from datetime import datetime

# same concept, different name per source -> canonical name (matches the TIS bundle's own wording)
CONCEPT_MAP = {
    "triglycerides": "Triglycerides", "triglyceride": "Triglycerides",
    "total cholesterol": "Total cholesterol", "cholesterol": "Total cholesterol",
    "hdl cholesterol": "HDL cholesterol",
    "non-hdl cholesterol": "Non-HDL cholesterol", "non hdl cholesterol": "Non-HDL cholesterol",
    "ldl cholesterol": "LDL cholesterol",
    "tc/hdl ratio": "TC:HDL ratio", "cholesterol:hdl ratio": "TC:HDL ratio", "tc:hdl ratio": "TC:HDL ratio",
    "body height": "Height", "height": "Height",
    "body weight": "Weight", "weight": "Weight",
    "body mass index (bmi) [ratio]": "BMI", "bmi": "BMI",
    "blood pressure": "Blood pressure",
}

# canonical item -> (target unit, {source unit: factor to target})
UNIT_FACTORS = {
    "Triglycerides":       ("mmol/L", {"mg/dl": 0.01129, "mmol/l": 1.0}),
    "Total cholesterol":   ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
    "HDL cholesterol":     ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
    "LDL cholesterol":     ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
    "Non-HDL cholesterol": ("mmol/L", {"mg/dl": 0.02586, "mmol/l": 1.0}),
}

def parse_tis_datetime(series):
    def parse_one(x):
        if pd.isna(x) or str(x).strip() == "":
            return pd.NaT
        digits = str(x).split(".")[0].zfill(14)  # pandas int inference can drop day's leading zero
        return datetime.strptime(digits, "%d%m%Y%H%M%S")
    return series.apply(parse_one)

In [ ]:
def normalize_long(df, origin):
    out = pd.DataFrame({
        "timestamp": parse_tis_datetime(df["Test or record date (UTC)"]),
        "From": df["From"],
        "canonical_item": df["Test or item"].str.strip().str.lower().map(CONCEPT_MAP).fillna(df["Test or item"]),
        "value_raw": pd.to_numeric(df["Result as recorded"], errors="coerce"),
        "unit_raw": df["Unit"].str.strip().str.lower(),
    })
    out["origin"] = origin

    def to_std(row):
        target, factors = UNIT_FACTORS.get(row["canonical_item"], (row["unit_raw"], {row["unit_raw"]: 1.0}))
        factor = factors.get(row["unit_raw"], 1.0)
        return pd.Series({"value_std": row["value_raw"] * factor, "unit_std": target})

    out[["value_std", "unit_std"]] = out.apply(to_std, axis=1)
    return out.drop(columns=["value_raw", "unit_raw"])


# wearable_stream.csv is wide (one row/day); reshape to the same long schema.
# Factors reverse-engineered by matching rows against the TIS bundle's "Wearable reading" entries.
WEARABLE_METRICS = {
    "recovery_score":     ("Recovery score", 1, "score"),
    "resting_heart_rate": ("Resting heart rate", 1, "bpm"),
    "hrv_rmssd":          ("VHR (RMSSD)", 1000, "ms"),           # seconds -> ms
    "quality_duration":   ("Hours asleep", 1 / 3_600_000, "h"),  # ms -> h
    "time_in_bed":        ("Time in bed", 1 / 3_600_000, "h"),   # ms -> h
}

def normalize_wearable(df):
    ts = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
    rows = [
        pd.DataFrame({
            "timestamp": ts, "From": "Wearable", "canonical_item": canonical_item,
            "value_std": df[col] * factor, "unit_std": unit, "origin": "Individual system",
        })
        for col, (canonical_item, factor, unit) in WEARABLE_METRICS.items()
    ]
    return pd.concat(rows, ignore_index=True)

In [ ]:
normalized = []
for df in individual.values():
    normalized.append(normalize_wearable(df) if "created_at" in df.columns else normalize_long(df, "Individual system"))
for df in tis_bundle.values():
    normalized.append(normalize_long(df, "TIS bundle"))

combined = pd.concat(normalized, ignore_index=True).dropna(subset=["timestamp"]).sort_values("timestamp")
combined.head()